In [1]:
!pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [2]:
!pip install lmdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.8/297.8 kB 8.6 MB/s eta 0:00:00


In [25]:
import os
import lmdb
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import zipfile
import random
import collections
import copy
import numpy as np

# Unzip the yeast_ppi.zip file
zip_path = "/content/yeast_ppi.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Augmentation dictionaries and functions
AMINO_ACID_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AMINO_ACID_LIST_TO_CODON_LIST = {
    'A': ['GCU', 'GCC', 'GCA', 'GCG'],
    'R': ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'N': ['AAU', 'AAC'],
    'D': ['GAU', 'GAC'],
    'C': ['UGU', 'UGC'],
    'Q': ['CAA', 'CAG'],
    'E': ['GAA', 'GAG'],
    'G': ['GGU', 'GGC', 'GGA', 'GGG'],
    'H': ['CAU', 'CAC'],
    'I': ['AUU', 'AUC', 'AUA'],
    'L': ['UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG'],
    'K': ['AAA', 'AAG'],
    'M': ['AUG'],
    'F': ['UUU', 'UUC'],
    'P': ['CCU', 'CCC', 'CCA', 'CCG'],
    'S': ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC'],
    'T': ['ACU', 'ACC', 'ACA', 'ACG'],
    'W': ['UGG'],
    'Y': ['UAU', 'UAC'],
    'V': ['GUU', 'GUC', 'GUA', 'GUG'],
    '*': ['UAA', 'UAG', 'UGA'],
}
CODON_LIST_TO_AMINO_ACID = {codon: aa for aa, codons in AMINO_ACID_LIST_TO_CODON_LIST.items() for codon in codons}

# Augmentation functions
def crop_random_segment(sequence, residue_len):
    seq_len = len(sequence)
    if seq_len == 0:
        return sequence
    crop_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - crop_len))
    return sequence[start:start + crop_len]

def delete_random_residues(sequence, residue_len):
    return [res for res in sequence if random.random() > residue_len]

def reverse_sequence(sequence, residue_len=None):
    return list(reversed(sequence))

def shuffle_random_segment(sequence, residue_len):
    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    seg_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - seg_len))
    segment = sequence[start:start + seg_len]
    random.shuffle(segment)
    sequence = sequence.copy()
    sequence[start:start + seg_len] = segment
    return sequence

def cut_and_shuffle(sequence, residue_len):
    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_cuts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_cuts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    random.shuffle(segments)
    return [res for seg in segments for res in seg]

def subsequence_shuffle(sequence, residue_len):
    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_parts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_parts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    selected_segments = random.sample(segments, min(len(segments), num_parts))
    return [res for seg in selected_segments for res in seg]

def insert_random_residues(sequence, residue_len):
    sequence = sequence.copy()
    seq_len = len(sequence)
    num_insertions = max(0, int(residue_len * seq_len))
    for _ in range(num_insertions):
        pos = random.randint(0, len(sequence))
        sequence.insert(pos, random.choice(AMINO_ACID_LIST))
    return sequence

def substitute_random_residues(sequence, residue_len):
    sequence = sequence.copy()
    seq_len = len(sequence)
    num_subs = max(0, int(residue_len * seq_len))
    for _ in range(num_subs):
        pos = random.randint(0, seq_len - 1)
        sequence[pos] = random.choice(AMINO_ACID_LIST)
    return sequence

def swap_random_residues(sequence, residue_len):
    sequence = sequence.copy()
    seq_len = len(sequence)
    num_swaps = max(0, int(residue_len * seq_len))
    for _ in range(num_swaps):
        if seq_len < 2:
            break
        i, j = random.sample(range(seq_len), 2)
        sequence[i], sequence[j] = sequence[j], sequence[i]
    return sequence

def back_translation_substitute(seq, residue_len):
    mRNA = []
    for aa in seq:
        if aa in AMINO_ACID_LIST_TO_CODON_LIST:
            mRNA.extend(list(random.choice(AMINO_ACID_LIST_TO_CODON_LIST[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(residue_len * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON_LIST_TO_AMINO_ACID.get(c, 'X')
            if aa in AMINO_ACID_LIST:
                aa_seq.append(aa)
    return aa_seq

# All augmentation functions in a list
AUGMENTATION_FUNCTIONS = [
    crop_random_segment,
    delete_random_residues,
    reverse_sequence,
    shuffle_random_segment,
    cut_and_shuffle,
    subsequence_shuffle,
    insert_random_residues,
    substitute_random_residues,
    swap_random_residues,
    back_translation_substitute
]

# Dataset class with augmentation
class LMDBProteinDataset(Dataset):
    def __init__(self, lmdb_path, max_length=512, augment=False,
                 augmentation_intensity=0.1, augmentation_prob=0.5, policy=None):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False)
        self.max_length = max_length
        self.augment = augment
        self.augmentation_intensity = augmentation_intensity
        self.augmentation_prob = augmentation_prob
        self.policy = policy
        self.valid_keys = []

        with self.env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                try:
                    data = pickle.loads(value)
                    if isinstance(data, dict) and all(k in data for k in ['primary_1', 'primary_2', 'interaction']):
                        self.valid_keys.append(key)
                except:
                    continue

        print(f"Loaded {len(self.valid_keys)} valid samples from {lmdb_path}")

    def __len__(self):
        return len(self.valid_keys)

    def __getitem__(self, idx):
        key = self.valid_keys[idx]
        with self.env.begin() as txn:
            data = pickle.loads(txn.get(key))

        primary_1 = data['primary_1']
        primary_2 = data['primary_2']

        # Apply augmentation to each sequence independently
        if self.augment:
            primary_1 = self._apply_augmentation(primary_1)
            primary_2 = self._apply_augmentation(primary_2)

        seq1 = self.encode_seq(primary_1)
        seq2 = self.encode_seq(primary_2)
        label = torch.tensor(data['interaction'], dtype=torch.float)
        return (seq1, seq2), label

    def _apply_augmentation(self, seq_str):
        """Apply random augmentation to a sequence string"""
        if self.policy:
            # Apply policy-based augmentation
            return self._apply_augmentation_policy(seq_str)
        else:
            # Apply standard augmentation
            return self._apply_standard_augmentation(seq_str)

    def _apply_standard_augmentation(self, seq_str):
        """Apply standard random augmentation"""
        if random.random() > self.augmentation_prob:
            return seq_str

        seq_list = list(seq_str)
        try:
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            augmented_list = aug_func(seq_list, self.augmentation_intensity)
            # Ensure we have at least 5 residues after augmentation
            if len(augmented_list) >= 5:
                return ''.join(augmented_list)
            return seq_str
        except Exception as e:
            print(f"Augmentation error: {str(e)}")
            return seq_str

    def _apply_augmentation_policy(self, seq_str):
        """Apply policy-based augmentation"""
        seq_list = list(seq_str)
        if not self.policy:
            return seq_str

        # Randomly select a sub-policy from the policy
        sub_policy = random.choice(self.policy)

        # Apply each operation in the sub-policy sequentially
        for operation in sub_policy:
            aug_func, p, lam = operation
            if random.random() < p:
                try:
                    augmented_list = aug_func(seq_list, lam)
                    # Skip if augmentation makes sequence too short
                    if len(augmented_list) < 5:
                        continue
                    seq_list = augmented_list
                except Exception as e:
                    print(f"Policy augmentation error: {str(e)}")

        return ''.join(seq_list)

    def encode_seq(self, seq):
        vocab = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        encoded = [vocab.get(aa, 0) for aa in seq[:self.max_length]]
        padded = encoded + [0] * (self.max_length - len(encoded))
        return torch.tensor(padded, dtype=torch.long)

# Collate function
def collate_fn(batch):
    seqs, labels = zip(*batch)
    seq1_batch, seq2_batch = zip(*seqs)
    return torch.stack(seq1_batch), torch.stack(seq2_batch), torch.stack(labels)

# LSTM Model
class ProteinInteractionLSTM(nn.Module):
    def __init__(self, embed_dim=64, hidden_dim=128, vocab_size=21):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return h_n[-1]  # use final hidden state

    def forward(self, seq1, seq2):
        h1 = self.encode(seq1)
        h2 = self.encode(seq2)
        combined = torch.cat([h1, h2], dim=1)
        return self.fc(combined).squeeze(1)

# Evaluation function
def evaluate(model, loader, device=DEVICE):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seq1, seq2, labels in loader:
            seq1, seq2, labels = seq1.to(device), seq2.to(device), labels.to(device)
            preds = model(seq1, seq2) > 0.5
            correct += (preds == labels.bool()).sum().item()
            total += labels.size(0)
    return correct / total

# Training function for Stage 1
def train_stage1(epochs=5):
    # Training dataset with uniform augmentation
    train_ds = LMDBProteinDataset(
        "/content/yeast_ppi/yeast_ppi_train.lmdb",
        augment=True,
        augmentation_intensity=0.1,
        augmentation_prob=0.7
    )

    # Validation and test without augmentation
    val_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_valid.lmdb")
    test_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_test.lmdb")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn)

    model = ProteinInteractionLSTM().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_acc = 0.0
    best_model = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seq1, seq2, labels in tqdm(train_loader, desc=f"Stage1 Epoch {epoch + 1}"):
            seq1, seq2, labels = seq1.to(DEVICE), seq2.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(seq1, seq2)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate(model, val_loader)

        print(f"Stage1 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model)

    # Test best model
    test_acc = evaluate(best_model, test_loader)
    print(f"Stage1 Test Accuracy: {test_acc:.4f}")

    return best_model

# Training function for Stage 2 (Policy Search)
def train_stage2(shared_model, num_policies=10, sub_policies_per_policy=5, finetune_epochs=1):
    # Create validation dataset (no augmentation)
    val_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_valid.lmdb")
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn)

    # Define candidate parameters
    p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    lambda_values = [0.05, 0.1, 0.2, 0.3, 0.4]

    # Generate random policies
    candidate_policies = []
    for _ in range(num_policies):
        policy = []
        for _ in range(sub_policies_per_policy):
            sub_policy = []
            # Each sub-policy has 2 operations
            for _ in range(2):
                aug_func = random.choice(AUGMENTATION_FUNCTIONS)
                p = random.choice(p_values)
                lam = random.choice(lambda_values)
                sub_policy.append((aug_func, p, lam))
            policy.append(sub_policy)
        candidate_policies.append(policy)

    best_policy = None
    best_val_acc = 0.0
    best_model = None

    print(f"Starting Stage 2: Testing {num_policies} policies with {finetune_epochs} finetune epochs each")

    for i, policy in enumerate(candidate_policies):
        print(f"\nEvaluating policy {i+1}/{num_policies}")

        # Create training dataset with current policy
        train_ds = LMDBProteinDataset(
            "/content/yeast_ppi/yeast_ppi_train.lmdb",
            augment=True,
            policy=policy
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)

        # Clone the shared model
        model = copy.deepcopy(shared_model)
        model.to(DEVICE)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # Smaller LR for fine-tuning

        # Fine-tune for a few epochs
        for epoch in range(finetune_epochs):
            model.train()
            total_loss = 0
            for seq1, seq2, labels in tqdm(train_loader, desc=f"Finetune Epoch {epoch+1}"):
                seq1, seq2, labels = seq1.to(DEVICE), seq2.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(seq1, seq2)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        # Evaluate on validation set
        val_acc = evaluate(model, val_loader)
        print(f"Policy {i+1} Val Acc: {val_acc:.4f}")

        # Update best policy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_policy = policy
            best_model = copy.deepcopy(model)

    return best_model, best_policy, best_val_acc

# Full APA training pipeline
def train_apa():
    # Stage 1: Train weight-shared model with uniform augmentation
    print("=" * 50)
    print("Starting Stage 1: Training weight-shared model")
    print("=" * 50)
    shared_model = train_stage1(epochs=5)

    # Stage 2: Policy search
    print("\n" + "=" * 50)
    print("Starting Stage 2: Policy search")
    print("=" * 50)
    best_model, best_policy, best_val_acc = train_stage2(
        shared_model,
        num_policies=10,
        sub_policies_per_policy=5,
        finetune_epochs=1
    )

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")

    # Evaluate on test set
    test_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_test.lmdb")
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn)
    test_acc = evaluate(best_model, test_loader)
    print(f"Final Test Accuracy: {test_acc:.4f}")

    return best_model, best_policy

# Run APA training
best_model, best_policy = train_apa()

Starting Stage 1: Training weight-shared model
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb
Loaded 95 valid samples from /content/yeast_ppi/yeast_ppi_valid.lmdb
Loaded 394 valid samples from /content/yeast_ppi/yeast_ppi_test.lmdb


Stage1 Epoch 1: 100%|██████████| 78/78 [00:04<00:00, 18.30it/s]


Stage1 Epoch 1/5 | Loss: 0.6932 | Val Acc: 0.4421


Stage1 Epoch 2: 100%|██████████| 78/78 [00:03<00:00, 24.78it/s]


Stage1 Epoch 2/5 | Loss: 0.6900 | Val Acc: 0.4632


...

Policy 8 Val Acc: 0.4421

Evaluating policy 9/10
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb


Finetune Epoch 1: 100%|██████████| 78/78 [00:03<00:00, 24.47it/s]


Policy 9 Val Acc: 0.4737

Evaluating policy 10/10
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb


Finetune Epoch 1: 100%|██████████| 78/78 [00:05<00:00, 14.22it/s]


Policy 10 Val Acc: 0.4632

Best validation accuracy: 0.4737
Loaded 394 valid samples from /content/yeast_ppi/yeast_ppi_test.lmdb
Final Test Accuracy: 0.4772


In [5]:
import os
import lmdb
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import zipfile
import random
import collections
import copy
import numpy as np

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Unzip the yeast_ppi.zip file
zip_path = "/content/yeast_ppi.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Augmentation dictionaries and functions
AMINO_ACID_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AMINO_ACID_LIST_TO_CODON_LIST = {
    'A': ['GCU', 'GCC', 'GCA', 'GCG'],
    'R': ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'N': ['AAU', 'AAC'],
    'D': ['GAU', 'GAC'],
    'C': ['UGU', 'UGC'],
    'Q': ['CAA', 'CAG'],
    'E': ['GAA', 'GAG'],
    'G': ['GGU', 'GGC', 'GGA', 'GGG'],
    'H': ['CAU', 'CAC'],
    'I': ['AUU', 'AUC', 'AUA'],
    'L': ['UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG'],
    'K': ['AAA', 'AAG'],
    'M': ['AUG'],
    'F': ['UUU', 'UUC'],
    'P': ['CCU', 'CCC', 'CCA', 'CCG'],
    'S': ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC'],
    'T': ['ACU', 'ACC', 'ACA', 'ACG'],
    'W': ['UGG'],
    'Y': ['UAU', 'UAC'],
    'V': ['GUU', 'GUC', 'GUA', 'GUG'],
    '*': ['UAA', 'UAG', 'UGA'],
}
CODON_LIST_TO_AMINO_ACID = {codon: aa for aa, codons in AMINO_ACID_LIST_TO_CODON_LIST.items() for codon in codons}

# Augmentation functions (with fixed random seeds for reproducibility within functions)
def crop_random_segment(sequence, residue_len):
    # Save current state and set seed
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len == 0:
        return sequence
    crop_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - crop_len))
    result = sequence[start:start + crop_len]

    # Restore random state
    random.setstate(state)
    return result

def delete_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    result = [res for res in sequence if random.random() > residue_len]

    random.setstate(state)
    return result

def reverse_sequence(sequence, residue_len=None):
    return list(reversed(sequence))

def shuffle_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    seg_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - seg_len))
    segment = sequence[start:start + seg_len]
    random.shuffle(segment)
    sequence = sequence.copy()
    sequence[start:start + seg_len] = segment

    random.setstate(state)
    return sequence

def cut_and_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_cuts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_cuts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    random.shuffle(segments)
    result = [res for seg in segments for res in seg]

    random.setstate(state)
    return result

def subsequence_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_parts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_parts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    selected_segments = random.sample(segments, min(len(segments), num_parts))
    result = [res for seg in selected_segments for res in seg]

    random.setstate(state)
    return result

def insert_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_insertions = max(0, int(residue_len * seq_len))
    for _ in range(num_insertions):
        pos = random.randint(0, len(sequence))
        sequence.insert(pos, random.choice(AMINO_ACID_LIST))

    random.setstate(state)
    return sequence

def substitute_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_subs = max(0, int(residue_len * seq_len))
    for _ in range(num_subs):
        pos = random.randint(0, seq_len - 1)
        sequence[pos] = random.choice(AMINO_ACID_LIST)

    random.setstate(state)
    return sequence

def swap_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_swaps = max(0, int(residue_len * seq_len))
    for _ in range(num_swaps):
        if seq_len < 2:
            break
        i, j = random.sample(range(seq_len), 2)
        sequence[i], sequence[j] = sequence[j], sequence[i]

    random.setstate(state)
    return sequence

def back_translation_substitute(seq, residue_len):
    state = random.getstate()
    random.seed(SEED)

    mRNA = []
    for aa in seq:
        if aa in AMINO_ACID_LIST_TO_CODON_LIST:
            mRNA.extend(list(random.choice(AMINO_ACID_LIST_TO_CODON_LIST[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(residue_len * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON_LIST_TO_AMINO_ACID.get(c, 'X')
            if aa in AMINO_ACID_LIST:
                aa_seq.append(aa)

    random.setstate(state)
    return aa_seq

# All augmentation functions in a list
AUGMENTATION_FUNCTIONS = [
    crop_random_segment,
    delete_random_residues,
    reverse_sequence,
    shuffle_random_segment,
    cut_and_shuffle,
    subsequence_shuffle,
    insert_random_residues,
    substitute_random_residues,
    swap_random_residues,
    back_translation_substitute
]

# Dataset class with augmentation
class LMDBProteinDataset(Dataset):
    def __init__(self, lmdb_path, max_length=512, augment=False,
                 augmentation_intensity=0.1, augmentation_prob=0.5, policy=None):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False)
        self.max_length = max_length
        self.augment = augment
        self.augmentation_intensity = augmentation_intensity
        self.augmentation_prob = augmentation_prob
        self.policy = policy
        self.valid_keys = []

        with self.env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                try:
                    data = pickle.loads(value)
                    if isinstance(data, dict) and all(k in data for k in ['primary_1', 'primary_2', 'interaction']):
                        self.valid_keys.append(key)
                except:
                    continue

        print(f"Loaded {len(self.valid_keys)} valid samples from {lmdb_path}")

    def __len__(self):
        return len(self.valid_keys)

    def __getitem__(self, idx):
        key = self.valid_keys[idx]
        with self.env.begin() as txn:
            data = pickle.loads(txn.get(key))

        primary_1 = data['primary_1']
        primary_2 = data['primary_2']

        # Apply augmentation to each sequence independently
        if self.augment:
            # Save and restore random state for dataset augmentation
            state = random.getstate()
            random.seed(SEED + idx)

            primary_1 = self._apply_augmentation(primary_1)
            primary_2 = self._apply_augmentation(primary_2)

            random.setstate(state)

        seq1 = self.encode_seq(primary_1)
        seq2 = self.encode_seq(primary_2)
        label = torch.tensor(data['interaction'], dtype=torch.float)
        return (seq1, seq2), label

    def _apply_augmentation(self, seq_str):
        """Apply random augmentation to a sequence string"""
        if self.policy:
            # Apply policy-based augmentation
            return self._apply_augmentation_policy(seq_str)
        else:
            # Apply standard augmentation
            return self._apply_standard_augmentation(seq_str)

    def _apply_standard_augmentation(self, seq_str):
        """Apply standard random augmentation"""
        if random.random() > self.augmentation_prob:
            return seq_str

        seq_list = list(seq_str)
        try:
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            augmented_list = aug_func(seq_list, self.augmentation_intensity)
            # Ensure we have at least 5 residues after augmentation
            if len(augmented_list) >= 5:
                return ''.join(augmented_list)
            return seq_str
        except Exception as e:
            print(f"Augmentation error: {str(e)}")
            return seq_str

    def _apply_augmentation_policy(self, seq_str):
        """Apply policy-based augmentation"""
        seq_list = list(seq_str)
        if not self.policy:
            return seq_str

        # Randomly select a sub-policy from the policy
        sub_policy = random.choice(self.policy)

        # Apply each operation in the sub-policy sequentially
        for operation in sub_policy:
            aug_func, p, lam = operation
            if random.random() < p:
                try:
                    augmented_list = aug_func(seq_list, lam)
                    # Skip if augmentation makes sequence too short
                    if len(augmented_list) < 5:
                        continue
                    seq_list = augmented_list
                except Exception as e:
                    print(f"Policy augmentation error: {str(e)}")

        return ''.join(seq_list)

    def encode_seq(self, seq):
        vocab = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        encoded = [vocab.get(aa, 0) for aa in seq[:self.max_length]]
        padded = encoded + [0] * (self.max_length - len(encoded))
        return torch.tensor(padded, dtype=torch.long)

# Collate function
def collate_fn(batch):
    seqs, labels = zip(*batch)
    seq1_batch, seq2_batch = zip(*seqs)
    return torch.stack(seq1_batch), torch.stack(seq2_batch), torch.stack(labels)

# LSTM Model
class ProteinInteractionLSTM(nn.Module):
    def __init__(self, embed_dim=64, hidden_dim=128, vocab_size=21):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return h_n[-1]  # use final hidden state

    def forward(self, seq1, seq2):
        h1 = self.encode(seq1)
        h2 = self.encode(seq2)
        combined = torch.cat([h1, h2], dim=1)
        return self.fc(combined).squeeze(1)

# Evaluation function
def evaluate(model, loader, device=DEVICE):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seq1, seq2, labels in loader:
            seq1, seq2, labels = seq1.to(device), seq2.to(device), labels.to(device)
            preds = model(seq1, seq2) > 0.5
            correct += (preds == labels.bool()).sum().item()
            total += labels.size(0)
    return correct / total

# Training function for Stage 1
def train_stage1(epochs=5):
    # Training dataset with uniform augmentation
    train_ds = LMDBProteinDataset(
        "/content/yeast_ppi/yeast_ppi_train.lmdb",
        augment=True,
        augmentation_intensity=0.1,
        augmentation_prob=0.7
    )

    # Validation and test without augmentation
    val_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_valid.lmdb")
    test_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_test.lmdb")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn)

    model = ProteinInteractionLSTM().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_acc = 0.0
    best_model = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seq1, seq2, labels in tqdm(train_loader, desc=f"Stage1 Epoch {epoch + 1}"):
            seq1, seq2, labels = seq1.to(DEVICE), seq2.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(seq1, seq2)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate(model, val_loader)

        print(f"Stage1 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model)

    # Test best model
    test_acc = evaluate(best_model, test_loader)
    print(f"Stage1 Test Accuracy: {test_acc:.4f}")

    return best_model

# Training function for Stage 2 (Policy Search)
def train_stage2(shared_model, num_policies=10, sub_policies_per_policy=5, finetune_epochs=1):
    # Create validation dataset (no augmentation)
    val_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_valid.lmdb")
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn)

    # Define candidate parameters
    p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    lambda_values = [0.05, 0.1, 0.2, 0.3, 0.4]

    # Generate random policies with fixed seed
    state = random.getstate()
    random.seed(SEED)

    candidate_policies = []
    for _ in range(num_policies):
        policy = []
        for _ in range(sub_policies_per_policy):
            sub_policy = []
            # Each sub-policy has 2 operations
            for _ in range(2):
                aug_func = random.choice(AUGMENTATION_FUNCTIONS)
                p = random.choice(p_values)
                lam = random.choice(lambda_values)
                sub_policy.append((aug_func, p, lam))
            policy.append(sub_policy)
        candidate_policies.append(policy)

    # Restore random state
    random.setstate(state)

    best_policy = None
    best_val_acc = 0.0
    best_model = None

    print(f"Starting Stage 2: Testing {num_policies} policies with {finetune_epochs} finetune epochs each")

    for i, policy in enumerate(candidate_policies):
        print(f"\nEvaluating policy {i+1}/{num_policies}")

        # Create training dataset with current policy
        train_ds = LMDBProteinDataset(
            "/content/yeast_ppi/yeast_ppi_train.lmdb",
            augment=True,
            policy=policy
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)

        # Clone the shared model
        model = copy.deepcopy(shared_model)
        model.to(DEVICE)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # Smaller LR for fine-tuning

        # Fine-tune for a few epochs
        for epoch in range(finetune_epochs):
            model.train()
            total_loss = 0
            for seq1, seq2, labels in tqdm(train_loader, desc=f"Finetune Epoch {epoch+1}"):
                seq1, seq2, labels = seq1.to(DEVICE), seq2.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(seq1, seq2)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        # Evaluate on validation set
        val_acc = evaluate(model, val_loader)
        print(f"Policy {i+1} Val Acc: {val_acc:.4f}")

        # Update best policy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_policy = policy
            best_model = copy.deepcopy(model)

    return best_model, best_policy, best_val_acc

# Full APA training pipeline
def train_apa():
    # Stage 1: Train weight-shared model with uniform augmentation
    print("=" * 50)
    print("Starting Stage 1: Training weight-shared model")
    print("=" * 50)
    shared_model = train_stage1(epochs=5)

    # Stage 2: Policy search
    print("\n" + "=" * 50)
    print("Starting Stage 2: Policy search")
    print("=" * 50)
    best_model, best_policy, best_val_acc = train_stage2(
        shared_model,
        num_policies=10,
        sub_policies_per_policy=5,
        finetune_epochs=1
    )

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")

    # Evaluate on test set
    test_ds = LMDBProteinDataset("/content/yeast_ppi/yeast_ppi_test.lmdb")
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn)
    test_acc = evaluate(best_model, test_loader)
    print(f"Final Test Accuracy: {test_acc:.4f}")

    return best_model, best_policy

# Run APA training
best_model, best_policy = train_apa()

Starting Stage 1: Training weight-shared model
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb
Loaded 95 valid samples from /content/yeast_ppi/yeast_ppi_valid.lmdb
Loaded 394 valid samples from /content/yeast_ppi/yeast_ppi_test.lmdb


Stage1 Epoch 1: 100%|██████████| 78/78 [00:05<00:00, 15.11it/s]


Stage1 Epoch 1/5 | Loss: 0.6932 | Val Acc: 0.4316


Stage1 Epoch 2: 100%|██████████| 78/78 [00:03<00:00, 23.64it/s]


Stage1 Epoch 2/5 | Loss: 0.6890 | Val Acc: 0.4211


...

Policy 8 Val Acc: 0.5158

Evaluating policy 9/10
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb


Finetune Epoch 1: 100%|██████████| 78/78 [00:03<00:00, 20.70it/s]


Policy 9 Val Acc: 0.5158

Evaluating policy 10/10
Loaded 4945 valid samples from /content/yeast_ppi/yeast_ppi_train.lmdb


Finetune Epoch 1: 100%|██████████| 78/78 [00:06<00:00, 12.46it/s]


Policy 10 Val Acc: 0.5263

Best validation accuracy: 0.5263
Loaded 394 valid samples from /content/yeast_ppi/yeast_ppi_test.lmdb
Final Test Accuracy: 0.4772


In [6]:
# Unzip the yeast_ppi.zip file
zip_path = "/content/remote_homology.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [8]:
import os
import lmdb
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import zipfile
import random
import collections
import copy
import numpy as np

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Augmentation dictionaries and functions
AMINO_ACID_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AMINO_ACID_LIST_TO_CODON_LIST = {
    'A': ['GCU', 'GCC', 'GCA', 'GCG'],
    'R': ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'N': ['AAU', 'AAC'],
    'D': ['GAU', 'GAC'],
    'C': ['UGU', 'UGC'],
    'Q': ['CAA', 'CAG'],
    'E': ['GAA', 'GAG'],
    'G': ['GGU', 'GGC', 'GGA', 'GGG'],
    'H': ['CAU', 'CAC'],
    'I': ['AUU', 'AUC', 'AUA'],
    'L': ['UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG'],
    'K': ['AAA', 'AAG'],
    'M': ['AUG'],
    'F': ['UUU', 'UUC'],
    'P': ['CCU', 'CCC', 'CCA', 'CCG'],
    'S': ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC'],
    'T': ['ACU', 'ACC', 'ACA', 'ACG'],
    'W': ['UGG'],
    'Y': ['UAU', 'UAC'],
    'V': ['GUU', 'GUC', 'GUA', 'GUG'],
    '*': ['UAA', 'UAG', 'UGA'],
}
CODON_LIST_TO_AMINO_ACID = {codon: aa for aa, codons in AMINO_ACID_LIST_TO_CODON_LIST.items() for codon in codons}

# Augmentation functions (with fixed random seeds for reproducibility within functions)
def crop_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len == 0:
        return sequence
    crop_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - crop_len))
    result = sequence[start:start + crop_len]

    random.setstate(state)
    return result

def delete_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    result = [res for res in sequence if random.random() > residue_len]

    random.setstate(state)
    return result

def reverse_sequence(sequence, residue_len=None):
    return list(reversed(sequence))

def shuffle_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    seg_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - seg_len))
    segment = sequence[start:start + seg_len]
    random.shuffle(segment)
    sequence = sequence.copy()
    sequence[start:start + seg_len] = segment

    random.setstate(state)
    return sequence

def cut_and_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_cuts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_cuts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    random.shuffle(segments)
    result = [res for seg in segments for res in seg]

    random.setstate(state)
    return result

def subsequence_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_parts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_parts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    selected_segments = random.sample(segments, min(len(segments), num_parts))
    result = [res for seg in selected_segments for res in seg]

    random.setstate(state)
    return result

def insert_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_insertions = max(0, int(residue_len * seq_len))
    for _ in range(num_insertions):
        pos = random.randint(0, len(sequence))
        sequence.insert(pos, random.choice(AMINO_ACID_LIST))

    random.setstate(state)
    return sequence

def substitute_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_subs = max(0, int(residue_len * seq_len))
    for _ in range(num_subs):
        pos = random.randint(0, seq_len - 1)
        sequence[pos] = random.choice(AMINO_ACID_LIST)

    random.setstate(state)
    return sequence

def swap_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_swaps = max(0, int(residue_len * seq_len))
    for _ in range(num_swaps):
        if seq_len < 2:
            break
        i, j = random.sample(range(seq_len), 2)
        sequence[i], sequence[j] = sequence[j], sequence[i]

    random.setstate(state)
    return sequence

def back_translation_substitute(seq, residue_len):
    state = random.getstate()
    random.seed(SEED)

    mRNA = []
    for aa in seq:
        if aa in AMINO_ACID_LIST_TO_CODON_LIST:
            mRNA.extend(list(random.choice(AMINO_ACID_LIST_TO_CODON_LIST[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(residue_len * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON_LIST_TO_AMINO_ACID.get(c, 'X')
            if aa in AMINO_ACID_LIST:
                aa_seq.append(aa)

    random.setstate(state)
    return aa_seq

# All augmentation functions in a list
AUGMENTATION_FUNCTIONS = [
    crop_random_segment,
    delete_random_residues,
    reverse_sequence,
    shuffle_random_segment,
    cut_and_shuffle,
    subsequence_shuffle,
    insert_random_residues,
    substitute_random_residues,
    swap_random_residues,
    back_translation_substitute
]

# Dataset class for Fold Classification
class FoldClassificationDataset(Dataset):
    def __init__(self, lmdb_path, max_length=512, augment=False,
                 augmentation_intensity=0.1, augmentation_prob=0.5, policy=None):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        self.max_length = max_length
        self.augment = augment
        self.augmentation_intensity = augmentation_intensity
        self.augmentation_prob = augmentation_prob
        self.policy = policy
        self.valid_keys = []
        self.num_classes = 1195  # Fold labels from 0 to 1194

        with self.env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                try:
                    data = pickle.loads(value)
                    if 'primary' in data and 'fold_label' in data:
                        self.valid_keys.append(key)
                except:
                    continue

        print(f"Loaded {len(self.valid_keys)} valid samples from {lmdb_path}")

    def __len__(self):
        return len(self.valid_keys)

    def __getitem__(self, idx):
        key = self.valid_keys[idx]
        with self.env.begin() as txn:
            data = pickle.loads(txn.get(key))

        primary = data['primary']
        fold_label = data['fold_label']

        # Apply augmentation to the sequence
        if self.augment:
            state = random.getstate()
            random.seed(SEED + idx)
            primary = self._apply_augmentation(primary)
            random.setstate(state)

        seq = self.encode_seq(primary)
        label = torch.tensor(fold_label, dtype=torch.long)
        return seq, label

    def _apply_augmentation(self, seq_str):
        if self.policy:
            return self._apply_augmentation_policy(seq_str)
        else:
            return self._apply_standard_augmentation(seq_str)

    def _apply_standard_augmentation(self, seq_str):
        if random.random() > self.augmentation_prob:
            return seq_str

        seq_list = list(seq_str)
        try:
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            augmented_list = aug_func(seq_list, self.augmentation_intensity)
            if len(augmented_list) >= 5:
                return ''.join(augmented_list)
            return seq_str
        except Exception as e:
            print(f"Augmentation error: {str(e)}")
            return seq_str

    def _apply_augmentation_policy(self, seq_str):
        seq_list = list(seq_str)
        if not self.policy:
            return seq_str

        sub_policy = random.choice(self.policy)

        for operation in sub_policy:
            aug_func, p, lam = operation
            if random.random() < p:
                try:
                    augmented_list = aug_func(seq_list, lam)
                    if len(augmented_list) < 5:
                        continue
                    seq_list = augmented_list
                except Exception as e:
                    print(f"Policy augmentation error: {str(e)}")

        return ''.join(seq_list)

    def encode_seq(self, seq):
        vocab = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        encoded = [vocab.get(aa, 0) for aa in seq[:self.max_length]]
        padded = encoded + [0] * (self.max_length - len(encoded))
        return torch.tensor(padded, dtype=torch.long)

# Collate function for classification
def collate_fn_classification(batch):
    seqs, labels = zip(*batch)
    return torch.stack(seqs), torch.stack(labels)

# LSTM Model for Classification
class ProteinFoldLSTM(nn.Module):
    def __init__(self, embed_dim=64, hidden_dim=128, vocab_size=21, num_classes=1195):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        # Concatenate forward and backward final hidden states
        h_n = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h_n)

# Evaluation function for classification
def evaluate_classification(model, loader, device=DEVICE):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(device), labels.to(device)
            outputs = model(seqs)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Training function for Stage 1 (Fold Classification)
def train_stage1_fold(epochs=10):
    # Training dataset with uniform augmentation
    train_ds = FoldClassificationDataset(
        "/content/remote_homology/remote_homology_train.lmdb",
        augment=True,
        augmentation_intensity=0.1,
        augmentation_prob=0.7
    )

    # Validation and test datasets
    val_ds = FoldClassificationDataset("/content/remote_homology/remote_homology_valid.lmdb")
    test_ds = FoldClassificationDataset("/content/remote_homology/remote_homology_test_fold_holdout.lmdb")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_classification)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_classification)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_classification)

    model = ProteinFoldLSTM(num_classes=train_ds.num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_acc = 0.0
    best_model = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seqs, labels in tqdm(train_loader, desc=f"Stage1 Epoch {epoch + 1}"):
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate_classification(model, val_loader)
        scheduler.step(val_acc)

        print(f"Stage1 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model)

    # Test best model
    test_acc = evaluate_classification(best_model, test_loader)
    print(f"Stage1 Test Accuracy: {test_acc:.4f}")

    return best_model

# Training function for Stage 2 (Policy Search for Fold Classification)
def train_stage2_fold(shared_model, num_policies=10, sub_policies_per_policy=5, finetune_epochs=3):
    val_ds = FoldClassificationDataset("/content/remote_homology/remote_homology_valid.lmdb")
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_classification)

    p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    lambda_values = [0.05, 0.1, 0.2, 0.3, 0.4]

    # Generate random policies with fixed seed
    state = random.getstate()
    random.seed(SEED)
    candidate_policies = []
    for _ in range(num_policies):
        policy = []
        for _ in range(sub_policies_per_policy):
            sub_policy = []
            for _ in range(2):
                aug_func = random.choice(AUGMENTATION_FUNCTIONS)
                p = random.choice(p_values)
                lam = random.choice(lambda_values)
                sub_policy.append((aug_func, p, lam))
            policy.append(sub_policy)
        candidate_policies.append(policy)
    random.setstate(state)

    best_policy = None
    best_val_acc = 0.0
    best_model = None

    print(f"Starting Stage 2: Testing {num_policies} policies with {finetune_epochs} finetune epochs each")

    for i, policy in enumerate(candidate_policies):
        print(f"\nEvaluating policy {i+1}/{num_policies}")

        train_ds = FoldClassificationDataset(
            "/content/remote_homology/remote_homology_train.lmdb",
            augment=True,
            policy=policy
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_classification)

        model = copy.deepcopy(shared_model)
        model.to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

        for epoch in range(finetune_epochs):
            model.train()
            total_loss = 0
            for seqs, labels in tqdm(train_loader, desc=f"Finetune Epoch {epoch+1}"):
                seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(seqs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        val_acc = evaluate_classification(model, val_loader)
        print(f"Policy {i+1} Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_policy = policy
            best_model = copy.deepcopy(model)

    return best_model, best_policy, best_val_acc

# Full APA training pipeline for Fold Classification
def train_apa_fold():
    print("=" * 50)
    print("Starting Stage 1: Training weight-shared model for Fold Classification")
    print("=" * 50)
    shared_model = train_stage1_fold(epochs=10)

    print("\n" + "=" * 50)
    print("Starting Stage 2: Policy search for Fold Classification")
    print("=" * 50)
    best_model, best_policy, best_val_acc = train_stage2_fold(
        shared_model,
        num_policies=10,
        sub_policies_per_policy=5,
        finetune_epochs=3
    )

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")

    # Evaluate on test set
    test_ds = FoldClassificationDataset("/content/remote_homology/remote_homology_test_fold_holdout.lmdb")
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_classification)
    test_acc = evaluate_classification(best_model, test_loader)
    print(f"Final Test Accuracy: {test_acc:.4f}")

    return best_model, best_policy

# Run APA training for Fold Classification
best_model, best_policy = train_apa_fold()

Starting Stage 1: Training weight-shared model for Fold Classification
Loaded 12312 valid samples from /content/remote_homology/remote_homology_train.lmdb
Loaded 736 valid samples from /content/remote_homology/remote_homology_valid.lmdb
Loaded 718 valid samples from /content/remote_homology/remote_homology_test_fold_holdout.lmdb


Stage1 Epoch 1: 100%|██████████| 193/193 [00:06<00:00, 29.28it/s]


Stage1 Epoch 1/10 | Loss: 5.8331 | Val Acc: 0.0408


Stage1 Epoch 2: 100%|██████████| 193/193 [00:05<00:00, 32.67it/s]


Stage1 Epoch 2/10 | Loss: 5.3705 | Val Acc: 0.0503


...

Policy 8 Val Acc: 0.0734

Evaluating policy 9/10
Loaded 12312 valid samples from /content/remote_homology/remote_homology_train.lmdb


Finetune Epoch 3: 100%|██████████| 193/193 [00:06<00:00, 30.71it/s]


Policy 9 Val Acc: 0.0707

Evaluating policy 10/10
Loaded 12312 valid samples from /content/remote_homology/remote_homology_train.lmdb


Finetune Epoch 3: 100%|██████████| 193/193 [00:07<00:00, 25.77it/s]


Policy 10 Val Acc: 0.0734

Best validation accuracy: 0.0788
Loaded 718 valid samples from /content/remote_homology/remote_homology_test_fold_holdout.lmdb
Final Test Accuracy: 0.0655


In [9]:
# Unzip the yeast_ppi.zip file
zip_path = "/content/subcellular_localization_2.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [10]:
import os
import lmdb
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import collections
import copy
import numpy as np

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Augmentation dictionaries and functions
AMINO_ACID_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AMINO_ACID_LIST_TO_CODON_LIST = {
    'A': ['GCU', 'GCC', 'GCA', 'GCG'],
    'R': ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'N': ['AAU', 'AAC'],
    'D': ['GAU', 'GAC'],
    'C': ['UGU', 'UGC'],
    'Q': ['CAA', 'CAG'],
    'E': ['GAA', 'GAG'],
    'G': ['GGU', 'GGC', 'GGA', 'GGG'],
    'H': ['CAU', 'CAC'],
    'I': ['AUU', 'AUC', 'AUA'],
    'L': ['UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG'],
    'K': ['AAA', 'AAG'],
    'M': ['AUG'],
    'F': ['UUU', 'UUC'],
    'P': ['CCU', 'CCC', 'CCA', 'CCG'],
    'S': ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC'],
    'T': ['ACU', 'ACC', 'ACA', 'ACG'],
    'W': ['UGG'],
    'Y': ['UAU', 'UAC'],
    'V': ['GUU', 'GUC', 'GUA', 'GUG'],
    '*': ['UAA', 'UAG', 'UGA'],
}
CODON_LIST_TO_AMINO_ACID = {codon: aa for aa, codons in AMINO_ACID_LIST_TO_CODON_LIST.items() for codon in codons}

# Augmentation functions (with fixed random seeds for reproducibility within functions)
def crop_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len == 0:
        return sequence
    crop_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - crop_len))
    result = sequence[start:start + crop_len]

    random.setstate(state)
    return result

def delete_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    result = [res for res in sequence if random.random() > residue_len]

    random.setstate(state)
    return result

def reverse_sequence(sequence, residue_len=None):
    return list(reversed(sequence))

def shuffle_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    seg_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - seg_len))
    segment = sequence[start:start + seg_len]
    random.shuffle(segment)
    sequence = sequence.copy()
    sequence[start:start + seg_len] = segment

    random.setstate(state)
    return sequence

def cut_and_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_cuts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_cuts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    random.shuffle(segments)
    result = [res for seg in segments for res in seg]

    random.setstate(state)
    return result

def subsequence_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_parts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_parts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    selected_segments = random.sample(segments, min(len(segments), num_parts))
    result = [res for seg in selected_segments for res in seg]

    random.setstate(state)
    return result

def insert_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_insertions = max(0, int(residue_len * seq_len))
    for _ in range(num_insertions):
        pos = random.randint(0, len(sequence))
        sequence.insert(pos, random.choice(AMINO_ACID_LIST))

    random.setstate(state)
    return sequence

def substitute_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_subs = max(0, int(residue_len * seq_len))
    for _ in range(num_subs):
        pos = random.randint(0, seq_len - 1)
        sequence[pos] = random.choice(AMINO_ACID_LIST)

    random.setstate(state)
    return sequence

def swap_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_swaps = max(0, int(residue_len * seq_len))
    for _ in range(num_swaps):
        if seq_len < 2:
            break
        i, j = random.sample(range(seq_len), 2)
        sequence[i], sequence[j] = sequence[j], sequence[i]

    random.setstate(state)
    return sequence

def back_translation_substitute(seq, residue_len):
    state = random.getstate()
    random.seed(SEED)

    mRNA = []
    for aa in seq:
        if aa in AMINO_ACID_LIST_TO_CODON_LIST:
            mRNA.extend(list(random.choice(AMINO_ACID_LIST_TO_CODON_LIST[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(residue_len * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON_LIST_TO_AMINO_ACID.get(c, 'X')
            if aa in AMINO_ACID_LIST:
                aa_seq.append(aa)

    random.setstate(state)
    return aa_seq

# All augmentation functions in a list
AUGMENTATION_FUNCTIONS = [
    crop_random_segment,
    delete_random_residues,
    reverse_sequence,
    shuffle_random_segment,
    cut_and_shuffle,
    subsequence_shuffle,
    insert_random_residues,
    substitute_random_residues,
    swap_random_residues,
    back_translation_substitute
]

# Dataset class for Binary Localization
class BinaryLocalizationDataset(Dataset):
    def __init__(self, lmdb_path, max_length=512, augment=False,
                 augmentation_intensity=0.1, augmentation_prob=0.5, policy=None):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        self.max_length = max_length
        self.augment = augment
        self.augmentation_intensity = augmentation_intensity
        self.augmentation_prob = augmentation_prob
        self.policy = policy
        self.valid_keys = []

        with self.env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                try:
                    data = pickle.loads(value)
                    if 'primary' in data and 'localization' in data:
                        self.valid_keys.append(key)
                except:
                    continue

        print(f"Loaded {len(self.valid_keys)} valid samples from {lmdb_path}")

    def __len__(self):
        return len(self.valid_keys)

    def __getitem__(self, idx):
        key = self.valid_keys[idx]
        with self.env.begin() as txn:
            data = pickle.loads(txn.get(key))

        primary = data['primary']
        localization = data['localization']

        # Apply augmentation to the sequence
        if self.augment:
            state = random.getstate()
            random.seed(SEED + idx)
            primary = self._apply_augmentation(primary)
            random.setstate(state)

        seq = self.encode_seq(primary)
        label = torch.tensor(localization, dtype=torch.float)  # Binary label
        return seq, label

    def _apply_augmentation(self, seq_str):
        if self.policy:
            return self._apply_augmentation_policy(seq_str)
        else:
            return self._apply_standard_augmentation(seq_str)

    def _apply_standard_augmentation(self, seq_str):
        if random.random() > self.augmentation_prob:
            return seq_str

        seq_list = list(seq_str)
        try:
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            augmented_list = aug_func(seq_list, self.augmentation_intensity)
            if len(augmented_list) >= 5:
                return ''.join(augmented_list)
            return seq_str
        except Exception as e:
            print(f"Augmentation error: {str(e)}")
            return seq_str

    def _apply_augmentation_policy(self, seq_str):
        seq_list = list(seq_str)
        if not self.policy:
            return seq_str

        sub_policy = random.choice(self.policy)

        for operation in sub_policy:
            aug_func, p, lam = operation
            if random.random() < p:
                try:
                    augmented_list = aug_func(seq_list, lam)
                    if len(augmented_list) < 5:
                        continue
                    seq_list = augmented_list
                except Exception as e:
                    print(f"Policy augmentation error: {str(e)}")

        return ''.join(seq_list)

    def encode_seq(self, seq):
        vocab = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        encoded = [vocab.get(aa, 0) for aa in seq[:self.max_length]]
        padded = encoded + [0] * (self.max_length - len(encoded))
        return torch.tensor(padded, dtype=torch.long)

# Collate function for binary classification
def collate_fn_binary(batch):
    seqs, labels = zip(*batch)
    return torch.stack(seqs), torch.stack(labels)

# LSTM Model for Binary Classification
class ProteinLocalizationLSTM(nn.Module):
    def __init__(self, embed_dim=64, hidden_dim=128, vocab_size=21):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        # Concatenate forward and backward final hidden states
        h_n = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h_n).squeeze(1)

# Evaluation function for binary classification
def evaluate_binary(model, loader, device=DEVICE):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(device), labels.to(device)
            outputs = model(seqs)
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Training function for Stage 1 (Binary Localization)
def train_stage1_binary(epochs=10):
    # Training dataset with uniform augmentation
    train_ds = BinaryLocalizationDataset(
        "/content/subcellular_localization_2/subcellular_localization_2_train.lmdb",
        augment=True,
        augmentation_intensity=0.1,
        augmentation_prob=0.7
    )

    # Validation and test datasets
    val_ds = BinaryLocalizationDataset("/content/subcellular_localization_2/subcellular_localization_2_valid.lmdb")
    test_ds = BinaryLocalizationDataset("/content/subcellular_localization_2/subcellular_localization_2_test.lmdb")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_binary)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_binary)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_binary)

    model = ProteinLocalizationLSTM().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_acc = 0.0
    best_model = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seqs, labels in tqdm(train_loader, desc=f"Stage1 Epoch {epoch + 1}"):
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate_binary(model, val_loader)
        scheduler.step(val_acc)

        print(f"Stage1 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model)

    # Test best model
    test_acc = evaluate_binary(best_model, test_loader)
    print(f"Stage1 Test Accuracy: {test_acc:.4f}")

    return best_model

# Training function for Stage 2 (Policy Search for Binary Localization)
def train_stage2_binary(shared_model, num_policies=10, sub_policies_per_policy=5, finetune_epochs=3):
    val_ds = BinaryLocalizationDataset("/content/subcellular_localization_2/subcellular_localization_2_valid.lmdb")
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_binary)

    p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    lambda_values = [0.05, 0.1, 0.2, 0.3, 0.4]

    # Generate random policies with fixed seed
    state = random.getstate()
    random.seed(SEED)
    candidate_policies = []
    for _ in range(num_policies):
        policy = []
        for _ in range(sub_policies_per_policy):
            sub_policy = []
            for _ in range(2):
                aug_func = random.choice(AUGMENTATION_FUNCTIONS)
                p = random.choice(p_values)
                lam = random.choice(lambda_values)
                sub_policy.append((aug_func, p, lam))
            policy.append(sub_policy)
        candidate_policies.append(policy)
    random.setstate(state)

    best_policy = None
    best_val_acc = 0.0
    best_model = None

    print(f"Starting Stage 2: Testing {num_policies} policies with {finetune_epochs} finetune epochs each")

    for i, policy in enumerate(candidate_policies):
        print(f"\nEvaluating policy {i+1}/{num_policies}")

        train_ds = BinaryLocalizationDataset(
            "/content/subcellular_localization_2/subcellular_localization_2_train.lmdb",
            augment=True,
            policy=policy
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_binary)

        model = copy.deepcopy(shared_model)
        model.to(DEVICE)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

        for epoch in range(finetune_epochs):
            model.train()
            total_loss = 0
            for seqs, labels in tqdm(train_loader, desc=f"Finetune Epoch {epoch+1}"):
                seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(seqs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        val_acc = evaluate_binary(model, val_loader)
        print(f"Policy {i+1} Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_policy = policy
            best_model = copy.deepcopy(model)

    return best_model, best_policy, best_val_acc

# Full APA training pipeline for Binary Localization
def train_apa_binary():
    print("=" * 50)
    print("Starting Stage 1: Training weight-shared model for Binary Localization")
    print("=" * 50)
    shared_model = train_stage1_binary(epochs=10)

    print("\n" + "=" * 50)
    print("Starting Stage 2: Policy search for Binary Localization")
    print("=" * 50)
    best_model, best_policy, best_val_acc = train_stage2_binary(
        shared_model,
        num_policies=10,
        sub_policies_per_policy=5,
        finetune_epochs=3
    )

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")

    # Evaluate on test set
    test_ds = BinaryLocalizationDataset("/content/subcellular_localization_2/subcellular_localization_2_test.lmdb")
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_binary)
    test_acc = evaluate_binary(best_model, test_loader)
    print(f"Final Test Accuracy: {test_acc:.4f}")

    return best_model, best_policy

# Run APA training for Binary Localization
best_model, best_policy = train_apa_binary()

Starting Stage 1: Training weight-shared model for Binary Localization
Loaded 5184 valid samples from /content/subcellular_localization_2/subcellular_localization_2_train.lmdb
Loaded 1729 valid samples from /content/subcellular_localization_2/subcellular_localization_2_valid.lmdb
Loaded 1749 valid samples from /content/subcellular_localization_2/subcellular_localization_2_test.lmdb


Stage1 Epoch 1: 100%|██████████| 81/81 [00:03<00:00, 23.32it/s]


Stage1 Epoch 1/10 | Loss: 0.6680 | Val Acc: 0.5876


Stage1 Epoch 2: 100%|██████████| 81/81 [00:02<00:00, 31.30it/s]


Stage1 Epoch 2/10 | Loss: 0.6520 | Val Acc: 0.4679


...

Policy 8 Val Acc: 0.6056

Evaluating policy 9/10
Loaded 5184 valid samples from /content/subcellular_localization_2/subcellular_localization_2_train.lmdb


Finetune Epoch 3: 100%|██████████| 81/81 [00:02<00:00, 29.97it/s]


Policy 9 Val Acc: 0.6073

Evaluating policy 10/10
Loaded 5184 valid samples from /content/subcellular_localization_2/subcellular_localization_2_train.lmdb


Finetune Epoch 3: 100%|██████████| 81/81 [00:03<00:00, 21.54it/s]


Policy 10 Val Acc: 0.6275

Best validation accuracy: 0.6657
Loaded 1749 valid samples from /content/subcellular_localization_2/subcellular_localization_2_test.lmdb
Final Test Accuracy: 0.6730


In [11]:
# Unzip the yeast_ppi.zip file
zip_path = "/content/subcellular_localization.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [12]:
import os
import lmdb
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import collections
import copy
import numpy as np

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Augmentation dictionaries and functions
AMINO_ACID_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AMINO_ACID_LIST_TO_CODON_LIST = {
    'A': ['GCU', 'GCC', 'GCA', 'GCG'],
    'R': ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'N': ['AAU', 'AAC'],
    'D': ['GAU', 'GAC'],
    'C': ['UGU', 'UGC'],
    'Q': ['CAA', 'CAG'],
    'E': ['GAA', 'GAG'],
    'G': ['GGU', 'GGC', 'GGA', 'GGG'],
    'H': ['CAU', 'CAC'],
    'I': ['AUU', 'AUC', 'AUA'],
    'L': ['UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG'],
    'K': ['AAA', 'AAG'],
    'M': ['AUG'],
    'F': ['UUU', 'UUC'],
    'P': ['CCU', 'CCC', 'CCA', 'CCG'],
    'S': ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC'],
    'T': ['ACU', 'ACC', 'ACA', 'ACG'],
    'W': ['UGG'],
    'Y': ['UAU', 'UAC'],
    'V': ['GUU', 'GUC', 'GUA', 'GUG'],
    '*': ['UAA', 'UAG', 'UGA'],
}
CODON_LIST_TO_AMINO_ACID = {codon: aa for aa, codons in AMINO_ACID_LIST_TO_CODON_LIST.items() for codon in codons}

# Augmentation functions (with fixed random seeds for reproducibility within functions)
def crop_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len == 0:
        return sequence
    crop_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - crop_len))
    result = sequence[start:start + crop_len]

    random.setstate(state)
    return result

def delete_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    result = [res for res in sequence if random.random() > residue_len]

    random.setstate(state)
    return result

def reverse_sequence(sequence, residue_len=None):
    return list(reversed(sequence))

def shuffle_random_segment(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    seg_len = max(1, int(residue_len * seq_len))
    start = random.randint(0, max(0, seq_len - seg_len))
    segment = sequence[start:start + seg_len]
    random.shuffle(segment)
    sequence = sequence.copy()
    sequence[start:start + seg_len] = segment

    random.setstate(state)
    return sequence

def cut_and_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_cuts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_cuts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    random.shuffle(segments)
    result = [res for seg in segments for res in seg]

    random.setstate(state)
    return result

def subsequence_shuffle(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    seq_len = len(sequence)
    if seq_len < 2:
        return sequence
    num_parts = max(1, int(residue_len * 10))
    cut_points = sorted(random.sample(range(1, seq_len), min(num_parts, seq_len-1))) + [seq_len]
    segments = [sequence[start:end] for start, end in zip([0] + cut_points[:-1], cut_points)]
    selected_segments = random.sample(segments, min(len(segments), num_parts))
    result = [res for seg in selected_segments for res in seg]

    random.setstate(state)
    return result

def insert_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_insertions = max(0, int(residue_len * seq_len))
    for _ in range(num_insertions):
        pos = random.randint(0, len(sequence))
        sequence.insert(pos, random.choice(AMINO_ACID_LIST))

    random.setstate(state)
    return sequence

def substitute_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_subs = max(0, int(residue_len * seq_len))
    for _ in range(num_subs):
        pos = random.randint(0, seq_len - 1)
        sequence[pos] = random.choice(AMINO_ACID_LIST)

    random.setstate(state)
    return sequence

def swap_random_residues(sequence, residue_len):
    state = random.getstate()
    random.seed(SEED)

    sequence = sequence.copy()
    seq_len = len(sequence)
    num_swaps = max(0, int(residue_len * seq_len))
    for _ in range(num_swaps):
        if seq_len < 2:
            break
        i, j = random.sample(range(seq_len), 2)
        sequence[i], sequence[j] = sequence[j], sequence[i]

    random.setstate(state)
    return sequence

def back_translation_substitute(seq, residue_len):
    state = random.getstate()
    random.seed(SEED)

    mRNA = []
    for aa in seq:
        if aa in AMINO_ACID_LIST_TO_CODON_LIST:
            mRNA.extend(list(random.choice(AMINO_ACID_LIST_TO_CODON_LIST[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(residue_len * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON_LIST_TO_AMINO_ACID.get(c, 'X')
            if aa in AMINO_ACID_LIST:
                aa_seq.append(aa)

    random.setstate(state)
    return aa_seq

# All augmentation functions in a list
AUGMENTATION_FUNCTIONS = [
    crop_random_segment,
    delete_random_residues,
    reverse_sequence,
    shuffle_random_segment,
    cut_and_shuffle,
    subsequence_shuffle,
    insert_random_residues,
    substitute_random_residues,
    swap_random_residues,
    back_translation_substitute
]

# Dataset class for Subcellular Localization (Multi-class)
class SubcellularLocalizationDataset(Dataset):
    def __init__(self, lmdb_path, max_length=512, augment=False,
                 augmentation_intensity=0.1, augmentation_prob=0.5, policy=None):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        self.max_length = max_length
        self.augment = augment
        self.augmentation_intensity = augmentation_intensity
        self.augmentation_prob = augmentation_prob
        self.policy = policy
        self.valid_keys = []
        self.num_classes = 10  # 10 subcellular localization classes

        with self.env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                try:
                    data = pickle.loads(value)
                    if 'primary' in data and 'localization' in data:
                        self.valid_keys.append(key)
                except:
                    continue

        print(f"Loaded {len(self.valid_keys)} valid samples from {lmdb_path}")

    def __len__(self):
        return len(self.valid_keys)

    def __getitem__(self, idx):
        key = self.valid_keys[idx]
        with self.env.begin() as txn:
            data = pickle.loads(txn.get(key))

        primary = data['primary']
        localization = data['localization']

        # Apply augmentation to the sequence
        if self.augment:
            state = random.getstate()
            random.seed(SEED + idx)
            primary = self._apply_augmentation(primary)
            random.setstate(state)

        seq = self.encode_seq(primary)
        label = torch.tensor(localization, dtype=torch.long)  # Multi-class label
        return seq, label

    def _apply_augmentation(self, seq_str):
        if self.policy:
            return self._apply_augmentation_policy(seq_str)
        else:
            return self._apply_standard_augmentation(seq_str)

    def _apply_standard_augmentation(self, seq_str):
        if random.random() > self.augmentation_prob:
            return seq_str

        seq_list = list(seq_str)
        try:
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            augmented_list = aug_func(seq_list, self.augmentation_intensity)
            if len(augmented_list) >= 5:
                return ''.join(augmented_list)
            return seq_str
        except Exception as e:
            print(f"Augmentation error: {str(e)}")
            return seq_str

    def _apply_augmentation_policy(self, seq_str):
        seq_list = list(seq_str)
        if not self.policy:
            return seq_str

        sub_policy = random.choice(self.policy)

        for operation in sub_policy:
            aug_func, p, lam = operation
            if random.random() < p:
                try:
                    augmented_list = aug_func(seq_list, lam)
                    if len(augmented_list) < 5:
                        continue
                    seq_list = augmented_list
                except Exception as e:
                    print(f"Policy augmentation error: {str(e)}")

        return ''.join(seq_list)

    def encode_seq(self, seq):
        vocab = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        encoded = [vocab.get(aa, 0) for aa in seq[:self.max_length]]
        padded = encoded + [0] * (self.max_length - len(encoded))
        return torch.tensor(padded, dtype=torch.long)

# Collate function for multi-class classification
def collate_fn_multiclass(batch):
    seqs, labels = zip(*batch)
    return torch.stack(seqs), torch.stack(labels)

# LSTM Model for Multi-class Classification
class ProteinSubcellularLSTM(nn.Module):
    def __init__(self, embed_dim=64, hidden_dim=128, vocab_size=21, num_classes=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        # Concatenate forward and backward final hidden states from last layer
        h_n = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h_n)

# Evaluation function for multi-class classification
def evaluate_multiclass(model, loader, device=DEVICE):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(device), labels.to(device)
            outputs = model(seqs)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Training function for Stage 1 (Subcellular Localization)
def train_stage1_subloc(epochs=15):
    # Training dataset with uniform augmentation
    train_ds = SubcellularLocalizationDataset(
        "/content/subcellular_localization/subcellular_localization_train.lmdb",
        augment=True,
        augmentation_intensity=0.1,
        augmentation_prob=0.7
    )

    # Validation and test datasets
    val_ds = SubcellularLocalizationDataset("/content/subcellular_localization/subcellular_localization_valid.lmdb")
    test_ds = SubcellularLocalizationDataset("/content/subcellular_localization/subcellular_localization_test.lmdb")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_multiclass)
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_multiclass)
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_multiclass)

    model = ProteinSubcellularLSTM(num_classes=train_ds.num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

    best_val_acc = 0.0
    best_model = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seqs, labels in tqdm(train_loader, desc=f"Stage1 Epoch {epoch + 1}"):
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate_multiclass(model, val_loader)
        scheduler.step(val_acc)

        print(f"Stage1 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model)
            print(f"New best model found at epoch {epoch+1} with val acc: {val_acc:.4f}")

    # Test best model
    test_acc = evaluate_multiclass(best_model, test_loader)
    print(f"Stage1 Test Accuracy: {test_acc:.4f}")

    return best_model

# Training function for Stage 2 (Policy Search for Subcellular Localization)
def train_stage2_subloc(shared_model, num_policies=10, sub_policies_per_policy=5, finetune_epochs=5):
    val_ds = SubcellularLocalizationDataset("/content/subcellular_localization/subcellular_localization_valid.lmdb")
    val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_fn_multiclass)

    p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    lambda_values = [0.05, 0.1, 0.2, 0.3, 0.4]

    # Generate random policies with fixed seed
    state = random.getstate()
    random.seed(SEED)
    candidate_policies = []
    for _ in range(num_policies):
        policy = []
        for _ in range(sub_policies_per_policy):
            sub_policy = []
            for _ in range(2):
                aug_func = random.choice(AUGMENTATION_FUNCTIONS)
                p = random.choice(p_values)
                lam = random.choice(lambda_values)
                sub_policy.append((aug_func, p, lam))
            policy.append(sub_policy)
        candidate_policies.append(policy)
    random.setstate(state)

    best_policy = None
    best_val_acc = 0.0
    best_model = None

    print(f"Starting Stage 2: Testing {num_policies} policies with {finetune_epochs} finetune epochs each")

    for i, policy in enumerate(candidate_policies):
        print(f"\nEvaluating policy {i+1}/{num_policies}")

        train_ds = SubcellularLocalizationDataset(
            "/content/subcellular_localization/subcellular_localization_train.lmdb",
            augment=True,
            policy=policy
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn_multiclass)

        model = copy.deepcopy(shared_model)
        model.to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

        for epoch in range(finetune_epochs):
            model.train()
            total_loss = 0
            for seqs, labels in tqdm(train_loader, desc=f"Finetune Epoch {epoch+1}"):
                seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(seqs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

        val_acc = evaluate_multiclass(model, val_loader)
        print(f"Policy {i+1} Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_policy = policy
            best_model = copy.deepcopy(model)
            print(f"New best policy found: Policy {i+1} with val acc: {val_acc:.4f}")

    return best_model, best_policy, best_val_acc

# Full APA training pipeline for Subcellular Localization
def train_apa_subloc():
    print("=" * 50)
    print("Starting Stage 1: Training weight-shared model for Subcellular Localization")
    print("=" * 50)
    shared_model = train_stage1_subloc(epochs=15)

    print("\n" + "=" * 50)
    print("Starting Stage 2: Policy search for Subcellular Localization")
    print("=" * 50)
    best_model, best_policy, best_val_acc = train_stage2_subloc(
        shared_model,
        num_policies=15,  # More policies for better exploration
        sub_policies_per_policy=5,
        finetune_epochs=5
    )

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")

    # Evaluate on test set
    test_ds = SubcellularLocalizationDataset("/content/subcellular_localization/subcellular_localization_test.lmdb")
    test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate_fn_multiclass)
    test_acc = evaluate_multiclass(best_model, test_loader)
    print(f"Final Test Accuracy: {test_acc:.4f}")

    # Save the best model
    torch.save({
        'model_state_dict': best_model.state_dict(),
        'policy': best_policy,
        'val_acc': best_val_acc,
        'test_acc': test_acc
    }, "best_subloc_model.pth")

    print("Model saved to best_subloc_model.pth")

    return best_model, best_policy

# Run APA training for Subcellular Localization
best_model, best_policy = train_apa_subloc()

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting Stage 1: Training weight-shared model for Subcellular Localization
Loaded 8420 valid samples from /content/subcellular_localization/subcellular_localization_train.lmdb
Loaded 2811 valid samples from /content/subcellular_localization/subcellular_localization_valid.lmdb
Loaded 2773 valid samples from /content/subcellular_localization/subcellular_localization_test.lmdb


Stage1 Epoch 1: 100%|██████████| 132/132 [00:08<00:00, 15.63it/s]


Stage1 Epoch 1/15 | Loss: 1.9060 | Val Acc: 0.4016
New best model found at epoch 1 with val acc: 0.4016


Stage1 Epoch 2: 100%|██████████| 132/132 [00:08<00:00, 15.61it/s]


...

Policy 13 Val Acc: 0.6297

Evaluating policy 14/15
Loaded 8420 valid samples from /content/subcellular_localization/subcellular_localization_train.lmdb


Finetune Epoch 5: 100%|██████████| 132/132 [00:08<00:00, 15.11it/s]


Policy 14 Val Acc: 0.6332

Evaluating policy 15/15
Loaded 8420 valid samples from /content/subcellular_localization/subcellular_localization_train.lmdb


Finetune Epoch 5: 100%|██████████| 132/132 [00:08<00:00, 14.78it/s]


Policy 15 Val Acc: 0.6279

Best validation accuracy: 0.6403
Loaded 2773 valid samples from /content/subcellular_localization/subcellular_localization_test.lmdb
Final Test Accuracy: 0.6076
Model saved to best_subloc_model.pth
